# CSharpRepl attache a un process .NET vivant

> Grain **DEEP/notebook-dotnet** — axe CSharpRepl de la serie *The Unexpected AI Stack: C#/.NET* (Part 1, Charles Chen 08/2026). Epic **#10473**, issue **#10802**. See #10473.

CSharpRepl est un REPL C# qui ne se contente pas d'executer du code dans un processus isole : il peut **s'attacher a une application .NET vivante** et **patcher ses methodes a chaud** — sans l'arreter, sans la redemarrer, sans recompiler.

Ce notebook demontre la capacite distinctive : un service de commandes tourne (avec `DOTNET_STARTUP_HOOKS`), le notebook s'y attache, lit son etat, **remplace puis enveloppe une methode a chaud**, et observe l'application changer de comportement *pendant qu'elle tourne*.

**Prerequis** : `dotnet tool install -g csharprepl` (le binaire est resolu automatiquement par le notebook).


## Pourquoi c'est non-trivial (parite Python)

| Capacite | Python | C#/.NET (cette demo) |
|---|---|---|
| Recharger du code | `%autoreload` (reimporte le module, ne touche pas aux objets existants) | **CSharpRepl** : `#replace` / `#wrap` sur une methode **deja chargee** |
| Attacher un debogueur | `pdb` (attache un debogueur, n'evalue pas dans le contexte du process) | `csharprepl connect <pid>` : evalue **dans** le process vivant, accede a ses statiques et services |
| Modifier le comportement a chaud | requiert de redeployer / reimporter | patch MonoMod applique **immediatement** et **persiste** jusqu'a `#revert` ou exit |

Le point distinctif : `%autoreload` recharge un *module* dans le process courant du notebook — il ne peut pas modifier une methode d'un *autre* process en cours d'execution. CSharpRepl le peut, via le hook de demarrage .NET.


## 1. Le hook de demarrage

`csharprepl connect init` imprime les deux variables d'environnement qui activent le connecteur dans l'application cible. On les pose **dans le shell qui lance l'application** (jamais system-wide).

Le programme de demonstration (`csharprepl-demo/`) est une application console qui boucle en imprimant le prix calcule par `OrderService.CalculatePrice(3, 10m)` toutes les 500 ms — exactement le genre de service qu'on ne veut pas arreter pour changer une regle de pricing.


In [1]:
// 0. Setup : helpers partages (l'etat persiste entre les cellules .NET Interactive)
#nullable enable
using System.Diagnostics;
using System.IO;
using System.Linq;
using System.Threading;
using System.Collections.Generic;

// Execute une commande et retourne stdout+stderr.
string Run(string fileName, string arguments)
{
    var psi = new ProcessStartInfo(fileName, arguments)
    {
        UseShellExecute = false,
        RedirectStandardOutput = true,
        RedirectStandardError = true,
        CreateNoWindow = true,
    };
    using var p = Process.Start(psi)!;
    var stdout = p.StandardOutput.ReadToEnd();
    var stderr = p.StandardError.ReadToEnd();
    p.WaitForExit();
    return stdout + (stderr.Length > 0 ? "\n[stderr] " + stderr : "");
}

// Resout CSharpRepl.exe dans le store dotnet tools global (version-agnostique).
string FindReplExe()
{
    var store = Path.Combine(
        Environment.GetFolderPath(Environment.SpecialFolder.UserProfile),
        ".dotnet", "tools", ".store", "csharprepl");
    if (!Directory.Exists(store)) throw new FileNotFoundException("store csharprepl introuvable: " + store);
    var exe = Directory.EnumerateFiles(store, "CSharpRepl.exe", SearchOption.AllDirectories)
        .FirstOrDefault(f => f.EndsWith(Path.Combine("net10.0", "win-x64", "CSharpRepl.exe")));
    if (exe is null) throw new FileNotFoundException("CSharpRepl.exe introuvable dans le store");
    return exe;
}

// Parse une ligne `$env:KEY = "value"` de la sortie de `connect init`.
string ParseEnvLine(string output, string key)
{
    var line = output.Split('\n').First(l => l.Contains("$env:" + key + " ="));
    return line.Split('=', 2)[1].Trim().Trim('"').Trim();
}

// Racine du repo (marchee ascendante depuis le repertoire courant).
string FindRepoRoot()
{
    var dir = new DirectoryInfo(Directory.GetCurrentDirectory());
    while (dir != null)
    {
        if (File.Exists(Path.Combine(dir.FullName, "MyIA.CoursIA.sln"))) return dir.FullName;
        dir = dir.Parent;
    }
    throw new DirectoryNotFoundException("racine du repo introuvable (MyIA.CoursIA.sln)");
}

string _replExe = FindReplExe();          // binaire CSharpRepl
string WorkingDir = FindRepoRoot();       // racine du repo
string Repl(string args) => Run(_replExe, args);

Process? _app = null;                     // process de l'application demo
int _pid = 0;                             // PID de l'application demo
List<string> _appLog = new();             // stdout/stderr capture de l'application demo

In [2]:
// 1a. Recuperer le chemin du hook (sortie de `connect init`, auto-detection machine)
var initOut = Run(_replExe, "connect init --shell pwsh");
var hook = ParseEnvLine(initOut, "DOTNET_STARTUP_HOOKS");
var hosting = ParseEnvLine(initOut, "ASPNETCORE_HOSTINGSTARTUPASSEMBLIES");
Console.WriteLine($"hook     : {Path.GetFileName(hook)}");
Console.WriteLine($"hosting  : {hosting}");

hook     : CSharpRepl.InjectedHook.dll


hosting  : CSharpRepl.InjectedHook


In [3]:
// 1b. Lancer l'application cible AVEC le hook (stdout capture -> _appLog)
var psi = new ProcessStartInfo("dotnet", "run --project MyIA.AI.Notebooks/GenAI/Vibe-Coding/docs/csharprepl-demo")
{
    UseShellExecute = false,
    RedirectStandardOutput = true,
    RedirectStandardError = true,
    WorkingDirectory = WorkingDir,
};
psi.Environment["DOTNET_STARTUP_HOOKS"] = hook;
psi.Environment["ASPNETCORE_HOSTINGSTARTUPASSEMBLIES"] = hosting;
_app = Process.Start(psi)!;
_app.OutputDataReceived += (_, e) => { if (e.Data is not null) lock (_appLog) _appLog.Add(e.Data); };
_app.ErrorDataReceived += (_, e) => { if (e.Data is not null) lock (_appLog) _appLog.Add("[err] " + e.Data); };
_app.BeginOutputReadLine();
_app.BeginErrorReadLine();

// Attendre la ligne "PID=" (le build `dotnet run` precede le demarrage)
var deadline = DateTime.UtcNow.AddSeconds(90);
while (DateTime.UtcNow < deadline)
{
    lock (_appLog)
    {
        var pidLine = _appLog.FirstOrDefault(l => l.Contains("PID="));
        if (pidLine is not null) { _pid = int.Parse(pidLine.Split("PID=")[1]); break; }
    }
    Thread.Sleep(200);
}
Console.WriteLine($"Application demarree, PID={_pid}");

Application demarree, PID=531940


## 2. Attacher le REPL au process vivant

`connect list` enumere les processus attachables. `connect <pid>` s'y attache ; on peut alors **evaluer des expressions dans le contexte du process** — acceder a ses types, ses statiques, ses services, ses donnees. Remarque : une expression **sans point-virgule** fait imprimer son resultat par le REPL.


In [4]:
// 2a. Lister les processus attachables
Console.WriteLine(Repl("connect list"));

                         
  PID    │ Process       
 ────────┼────────────── 
  531940 │ LiveOrderApp  
  541428 │ dotnet        
                         
Connect with csharprepl connect <PID>.
Hint: you most likely want to connect to the 'LiveOrderApp' process (PID 
531940).




In [5]:
// 2b. Evaluer DANS le process vivant : appeler la methode du service
// (le process n'est ni arrete ni redemarre)
Console.WriteLine(Repl($"connect {_pid} -e \"LiveOrderApp.OrderService.CalculatePrice(5, 7m)\""));

35



## 3. Lire l'etat vivant du process

On peut aussi lire les **statiques** du process — l'etat reel, pas une copie.


In [6]:
// 3. Lire un compteur statique vivant (ici : nombre de calculs effectues)
Console.WriteLine(Repl($"connect {_pid} -e \"LiveOrderApp.OrderService.ComputeCount\""));

6



## 4. Patcher une methode a chaud (`#replace`)

On definit d'abord une **methode de remplacement** (meme signature que la cible), puis `#replace` l'applique au process **immediatement**. L'application continue de tourner — et son comportement change.


In [7]:
// 4a. Definir la methode de remplacement (remise de 20%)
// (signature identique : int, decimal -> decimal)
Console.WriteLine(Repl($"connect {_pid} -e \"decimal salePrice(int q, decimal u) => q * u * 0.8m;\""));

In [8]:
// 4b. Appliquer le patch a chaud
Console.WriteLine(Repl($"connect {_pid} -e \"#replace LiveOrderApp.OrderService.CalculatePrice with salePrice\""));

patched static Decimal LiveOrderApp.OrderService.CalculatePrice(Int32 quantity, Decimal unitPrice)  ←  salePrice  (patch #1)



In [9]:
// 4c. Observer le changement dans l'application VIVANTE (log capture en memoire)
Thread.Sleep(1200);
lock (_appLog)
{
    var tail = _appLog.Where(l => l.Contains("price=")).TakeLast(3);
    Console.WriteLine(string.Join(Environment.NewLine, tail));
}

[7] price=30
[8] price=24,0
[9] price=24,0


## 5. Envelopper (`#wrap`), inventaire (`#patches`), annuler (`#revert`)

`#wrap` enveloppe la methode courante : le wrapper recoit `orig` (le delegate original) en premier parametre — c'est le pattern documente pour le wrapping d'une methode statique. `#patches` liste les patches actifs ; `#revert all` les annule tous.


In [10]:
// 5a. Definir le wrapper de journalisation (le delegate `orig` vient en premier parametre)
// (les `\\\"` sont l'echappement CommandLineToArgvW : quotes litterales dans l'argv du process cible)
Console.WriteLine(Repl($"connect {_pid} -e \"decimal logged(Func<int, decimal, decimal> orig, int q, decimal u) {{ System.Console.WriteLine(\\\"repl-log: \\\" + q + \\\" x \\\" + u); return orig(q, u); }}\""));

In [11]:
// 5b. Appliquer le wrap
Console.WriteLine(Repl($"connect {_pid} -e \"#wrap LiveOrderApp.OrderService.CalculatePrice with logged\""));

wrapped static Decimal LiveOrderApp.OrderService.CalculatePrice(Int32 quantity, Decimal unitPrice)  ←  logged  (patch #2)



In [12]:
// 5c. Inventaire des patches actifs
Console.WriteLine(Repl($"connect {_pid} -e \"#patches\""));

  #1  [Replace]  static Decimal LiveOrderApp.OrderService.CalculatePrice(Int32 quantity, Decimal unitPrice)  ←  salePrice
  #2  [Wrap]  static Decimal LiveOrderApp.OrderService.CalculatePrice(Int32 quantity, Decimal unitPrice)  ←  logged



In [13]:
// 5d. Annuler tous les patches -> l'application revient au comportement original
Console.WriteLine(Repl($"connect {_pid} -e \"#revert all\""));

reverted 2 patch(es).



## Exercice 1 : patch `#replace` d'un calcul de TVA

L'application `LiveOrderApp` calcule `price = quantity * unitPrice` (TVA incluse a 20%). Ajoutez une methode `withVat` qui applique 20% de TVA en plus, puis `#replace` `OrderService.CalculatePrice`. L'application doit commencer a imprimer `price=36` (3 × 10 × 1,2) au lieu de `price=30`.

**Indice** : reprenez exactement le pattern de la section 4 (methode de meme signature, puis `#replace`).


In [14]:
// Exercice a completer : definir withVat puis #replace OrderService.CalculatePrice
// decimal withVat(int q, decimal u) => ...;
// #replace LiveOrderApp.OrderService.CalculatePrice with withVat
// Verifier dans le log que price passe a 36.
Console.WriteLine("Exercice a completer");

Exercice a completer


## Exercice 2 : `#wrap` avec journalisation

Enveloppez `OrderService.CalculatePrice` avec un wrapper qui **journalise chaque appel** (quantite + prix unitaire) avant de deleguer a l'original — puis `#patches` pour verifier que le wrap est actif.

**Indice** : le wrapper recoit `orig` (le delegate original) en premier parametre.


In [15]:
// Exercice a completer : wrapper de journalisation puis #wrap
// decimal logged(Func<int, decimal, decimal> orig, int q, decimal u) { ... ; return orig(q, u); }
// #wrap LiveOrderApp.OrderService.CalculatePrice with logged
Console.WriteLine("Exercice a completer");

Exercice a completer


## Exercice 3 : lecture d'etat vivant multi-cible

La liste `connect list` peut montrer plusieurs processus attachables (par ex. le kernel .NET Interactive lui-meme). Ecrivez un snippet qui : (1) liste les processus, (2) s'attache a celui de `LiveOrderApp`, (3) lit le compteur statique `OrderService.ComputeCount` a deux instants distants de 1,5 s pour constater qu'il augmente — preuve que le process vit et que le REPL lit son etat reel.

**Indice** : `connect <pid> -e "..."` retourne le resultat de l'expression (sans point-virgule).


In [16]:
// Exercice a completer : mesurer ComputeCount a deux instants
// var c1 = int.Parse(Repl($"connect {_pid} -e \"LiveOrderApp.OrderService.ComputeCount\"").Trim());
// Thread.Sleep(1500);
// var c2 = ...;
// Console.WriteLine($"count: {c1} -> {c2}");
Console.WriteLine("Exercice a completer");

Exercice a completer


## 6. Arret propre

On detache le REPL (`exit` ou EOF) et on arrete l'application de demonstration.


In [17]:
// 6. Arreter proprement l'application vivante
_app?.Kill(entireProcessTree: true);
Console.WriteLine("Application arretee.");

Application arretee.


## Conclusion

Nous avons demontre, **avec du code execute** : CSharpRepl s'attache a un process .NET vivant, y evalue des expressions, lit son etat, **patche une methode a chaud** (`#replace`), l'**enveloppe** (`#wrap`) et **annule** (`#revert`) — l'application changeant de comportement en continu, sans arret ni redemarrage.

C'est la ligne « Introspection d'un process vivant » de la table de parite de l'Epic **#10473** : ce que `%autoreload` / `pdb` ne font pas cote Python, le REPL attache le fait nativement cote .NET — la verification et la modification se font **dans** le process, pas en post-processing.

*Rappel securite* : `csharprepl connect` equivaut a executer du code arbitraire dans le process cible, avec ses privileges. Outil de developpement et de diagnostic — jamais sur un process de production.
